In [1]:
import os
import multiprocessing
import pandas as pd
import numpy as np
from pathlib import Path
from joblib import Parallel, delayed

#   종목 설정
code_001 = '005930'

#   경로 설정
path_001 = r'./data'

#   병렬 처리
cl = multiprocessing.cpu_count() - 1

#   경로 내 해당 패턴 파일 확인
df_list_001 = list(Path(path_001).rglob('df_tokenized*.pkl'))
print(df_list_001)

[WindowsPath('data/df_tokenized_260102_260331.pkl'), WindowsPath('data/df_tokenized_260401_260604.pkl')]


In [2]:
#   pickle 파일 일괄 처리 함수 생성
def read_pkl_001(file_01):
    return pd.read_pickle(file_01)

#   경로 내 파일 list 형태로 호출
df_ls_001 = Parallel(n_jobs = cl)(delayed(read_pkl_001)(file_01) for file_01 in df_list_001)

In [3]:
#   list형태의 파일 호출값 확인
print(type(df_ls_001)), print(len(df_ls_001))

#   list 형태의 파일에 대해 데이터 프레임으로 결합
df_001 = pd.concat(df_ls_001, axis = 0, ignore_index = True)
print(df_001.head())

<class 'list'>
2
    섹션                                                 제목     언론사  \
0  258  잘 나가는 방위산업株…한화에어로·LIG넥스원, 나란히 AA로 신용 ‘레벨업’ [투자...   헤럴드경제   
1  263                   최태원 SK 회장 “AI 시대, ‘승풍파랑’ 도전 나서자”    중앙일보   
2  263                               누적 수익률 610만%…역사로 남았다  한국경제TV   
3  263                            韓참여 국제연구진 ‘나홀로 행성’ 첫 발견    문화일보   
4  263                     오세훈 서울시장 “비상계엄 등 잘못 인정하고 반성해야”  이코노미스트   

                                                  본문 Target_Date  \
0  2025년 활약한 방위산업株, 신용평가 등급 A+~AA 포진 한화에어로·LIG넥스원...  2026-01-02   
1  “인공지능(AI)이라는 거대한 변화의 바람을 타고 글로벌 시장의 거친 파도를 거침없...  2026-01-02   
2  '오마하의 현인' 버핏 은퇴 새해 첫날 부회장에게 버크셔 CEO 자리 넘겨 미국의 ...  2026-01-02   
3  1만광년 거리 토성급 행성 한국 연구진이 참여한 국제 공동연구진이 우리나라의 외계행...  2026-01-02   
4  국민의힘 계엄 반성 등 과거와의 단절 요구 ‘국민이 먹고 사는 문제’ 집중해야 한다...  2026-01-02   

                                                  수정  \
0  2025년 활약한 방위산업   신용평가 등급 A  AA 포진 한화에어로 LIG넥스원...   
1  인공지능 AI 이라는 거대한 변화의 바람을 타고 글로벌 시장의 거친 파도를 거침

In [4]:
#   정답 데이터(삼성전자 주가 데이터) 추출(yfinance 이용)
answer_list_001 = list(Path(path_001).rglob(f'*{code_001}*.csv'))
answer_list_002 = answer_list_001[-1]
print(answer_list_002)

answer_df_001 = pd.read_csv(answer_list_002)
answer_df_001.head()

data\005930.KS_raw_data_251222_260605.csv


,Price,Close,High,Low,Open,Volume
0,Ticker,005930.KS,005930.KS,005930.KS,005930.KS,005930.KS
1,Date,NaN,NaN,NaN,NaN,NaN
2,2025-12-22,109738.3046875,109738.3046875,108546.57649179864,108943.81922369909,24859171
3,2025-12-23,110731.4140625,111724.52091507846,109638.99652466367,110135.54995095292,20419187
4,2025-12-24,110334.1640625,111625.20288591359,110135.54270505176,111625.20288591359,12492939


In [5]:
"""데이터 전처리"""
#   열 이름 변경
answer_col_001 = answer_df_001.columns.tolist()
answer_col_001[0] = 'Target_Date'
answer_df_001.columns = answer_col_001
answer_col_001

#   1행 및 2행 제거
answer_df_001 = answer_df_001.drop([0, 1], axis = 0)
answer_df_001.head(), answer_df_001.tail()

(  Target_Date           Close                High                 Low  \
 2  2025-12-22  109738.3046875      109738.3046875  108546.57649179864   
 3  2025-12-23  110731.4140625  111724.52091507846  109638.99652466367   
 4  2025-12-24  110334.1640625  111625.20288591359  110135.54270505176   
 5  2025-12-26  116193.4921875      116193.4921875  111625.20104166666   
 6  2025-12-29   119253.171875  119452.75877353556  117955.85703451883   
 
                  Open    Volume  
 2  108943.81922369909  24859171  
 3  110135.54995095292  20419187  
 4  111625.20288591359  12492939  
 5  111625.20104166666  34018174  
 6  119153.37842573223  19676004  ,
     Target_Date     Close      High       Low      Open    Volume
 105  2026-05-28  299500.0  306500.0  287500.0  305000.0  30195334
 106  2026-05-29  317000.0  319000.0  305500.0  309500.0  37241537
 107  2026-06-01  349000.0  354500.0  319500.0  319500.0  45052488
 108  2026-06-02  360500.0  370000.0  342000.0  360500.0  44720282
 109  20

In [9]:
print(answer_df_001.info())

#   errors = 'coerce': 에러가 출력되면 무시 + 강제로 결측 처리
answer_df_001['Target_Date']  = pd.to_datetime(answer_df_001['Target_Date'], errors = 'coerce').astype('datetime64[ns]')
answer_df_001['Close'] = pd.to_numeric(answer_df_001['Close'], errors = 'coerce')
answer_df_001['High']  = pd.to_numeric(answer_df_001['High'], errors = 'coerce')
answer_df_001['Low']  = pd.to_numeric(answer_df_001['Low'], errors = 'coerce')
answer_df_001['Open']  = pd.to_numeric(answer_df_001['Open'], errors = 'coerce')
answer_df_001['Volume']  = pd.to_numeric(answer_df_001['Volume'], errors = 'coerce')

print(answer_df_001.info())

<class 'pandas.DataFrame'>
RangeIndex: 108 entries, 2 to 109
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Target_Date  108 non-null    str  
 1   Close        108 non-null    str  
 2   High         108 non-null    str  
 3   Low          108 non-null    str  
 4   Open         108 non-null    str  
 5   Volume       108 non-null    str  
dtypes: str(6)
memory usage: 5.2 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 108 entries, 2 to 109
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Target_Date  108 non-null    datetime64[ns]
 1   Close        108 non-null    float64       
 2   High         108 non-null    float64       
 3   Low          108 non-null    float64       
 4   Open         108 non-null    float64       
 5   Volume       108 non-null    int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage:

In [10]:
"""answer열 생성"""
answer_df_002 = answer_df_001.copy()

#   05이평선 생성
answer_df_002['MA5'] = answer_df_002['Close'].rolling(window = 5).mean()

#   5거래일 이후 이평선 현재 행으로 이동
answer_df_002['pro_MA5'] = answer_df_002['MA5'].shift(-5)

#   answer열 생성: 거래당일 이평선 및 5거래일 이후 이평선 비교
#   answer: 5거래일 이후 이평선 당일 이평선보다 3% 이상 증가
answer_df_002['answer'] = ((answer_df_002['pro_MA5'] / answer_df_002['MA5']) - 1) >= 0.03

print(f"거래일 중복 여부: {sum(answer_df_002['Target_Date'].duplicated())}")
answer_df_002['answer'].value_counts()

거래일 중복 여부: 0


answer
True     67
False    41
Name: count, dtype: int64

In [13]:
"""데이터 병합"""
#   기준열(날짜) 데이터 형식 일치
df_001['Target_Date'] = pd.to_datetime(df_001['Target_Date'], errors = 'coerce').astype('datetime64[ns]')

answer_df_003 = pd.DataFrame(answer_df_002[['Target_Date', 'answer']])

df_001 = df_001.sort_values('Target_Date')
answer_df_003 = answer_df_003.sort_values('Target_Date')


#   기존 뉴스기사 토큰화 파일에 정답 열 병합
df_002 = pd.merge(
    df_001, answer_df_003, on = 'Target_Date', how = 'inner'
)

print(df_002.info())

<class 'pandas.DataFrame'>
RangeIndex: 651361 entries, 0 to 651360
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   섹션           651361 non-null  int64         
 1   제목           651361 non-null  object        
 2   언론사          651361 non-null  object        
 3   본문           651361 non-null  object        
 4   Target_Date  651361 non-null  datetime64[ns]
 5   수정           651361 non-null  object        
 6   tokens       651361 non-null  object        
 7   answer       651361 non-null  bool          
dtypes: bool(1), datetime64[ns](1), int64(1), object(5)
memory usage: 35.4+ MB
None


In [14]:
#   지정 일자별 파일 분리 함수 생성
def divide_by_date(df_01, start_date, end_date, col_01 = 'Target_Date'):
    #   날짜 범위별 파일 생성
    df_02 = df_01[(df_01[col_01] >= start_date) & (df_01[col_01] < end_date)]

    #   범위별 파일명 지정
    df_02[col_01] = pd.to_datetime(df_02[col_01])

    min_date = df_02[col_01].min().strftime('%y%m%d')
    max_date = df_02[col_01].max().strftime('%y%m%d')

    #   경로 지정 및 파일 저장
    file_01 = os.path.join('.',  f'df_{code_001}_{min_date}_{max_date}.pkl')
    df_02.to_pickle(file_01)

    return df_02


In [15]:
#   파일 분리 및 파일 저장
df_until_MAR  = divide_by_date(df_002, '2026-01-01', '2026-04-01')
df_until_JUNE  = divide_by_date(df_002, '2026-04-01', '2026-07-01')

In [16]:
#   파일 생성 확인
list(Path('./').rglob(f'*{code_001}*.pkl'))

[WindowsPath('df_005930_260102_260331.pkl'),
 WindowsPath('df_005930_260401_260604.pkl')]